# 10 - Operational Prioritization

## Objective

Use the frozen XGBoost model from Notebook 08 to score a reproducible
November–December operational sample, rank flights under limited review
capacity, and answer **RQ5**.

**RQ5:** Can optimization identify more actually delayed flights than random
selection or simple rule-based prioritization when operational capacity is
limited?

At every capacity, all strategies are compared using the same number of selected
flights. Random selection is repeated 500 times. No model, threshold, feature,
or hyperparameter is changed in this notebook.


## Load Project Configuration and Libraries


In [1]:
from __future__ import annotations

import gc
import importlib.util
import io
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
from databricks.sdk import WorkspaceClient
from dotenv import dotenv_values
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not bootstrap.is_file():
    bootstrap = Path.cwd() / "import_path.py"

spec = importlib.util.spec_from_file_location("import_path", bootstrap)
import_path = importlib.util.module_from_spec(spec)
spec.loader.exec_module(import_path)

from config import project_config as cfg
from utils.operational_prioritization import (
    add_operational_scores,
    build_ranking_table,
    compare_prioritization_strategies,
)

DATABRICKS_PROFILE = os.getenv(
    "DATABRICKS_CONFIG_PROFILE",
    "capstone-serverless",
)
api_env_candidates = [
    Path.cwd() / "api" / ".env",
    Path.cwd().parent / "api" / ".env",
]
api_env_path = next(
    (candidate for candidate in api_env_candidates if candidate.is_file()),
    None,
)
api_credentials = dotenv_values(api_env_path) if api_env_path else {}
LOCAL_DATABRICKS_HOST = str(
    api_credentials.get("DATABRICKS_SERVER_HOSTNAME", "")
).strip()
LOCAL_DATABRICKS_TOKEN = str(
    api_credentials.get("DATABRICKS_ACCESS_TOKEN", "")
).strip()
if LOCAL_DATABRICKS_HOST and not LOCAL_DATABRICKS_HOST.startswith("http"):
    LOCAL_DATABRICKS_HOST = f"https://{LOCAL_DATABRICKS_HOST}"

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    builder = DatabricksSession.builder.serverless()
    if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
        builder = builder.host(LOCAL_DATABRICKS_HOST).token(
            LOCAL_DATABRICKS_TOKEN
        )
    else:
        builder = builder.profile(DATABRICKS_PROFILE)
    spark = builder.getOrCreate()

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

print("Configuration, Spark session, and prioritization libraries loaded.")
print(f"Operational window: {cfg.SCORING_START_DATE} to {cfg.SCORING_END_DATE}")


/Users/daniel.montero/Documents/Master of Data Analytics/UNFC Term 5 Capstone/AI-Powered-Flight-Delay-Risk-Prediction-and-Operational-Prioritization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration, Spark session, and prioritization libraries loaded.
Operational window: 2025-11-01 to 2025-12-31


## Load and Validate the Frozen Final Model

Notebook 10 consumes the fitted model and preprocessing pipeline saved by
Notebook 08. Notebook 09 must also have saved the SHAP feature labels used by
the dashboard.


In [2]:
def artifact_client() -> WorkspaceClient:
    try:
        dbutils
    except NameError:
        if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
            return WorkspaceClient(
                host=LOCAL_DATABRICKS_HOST,
                token=LOCAL_DATABRICKS_TOKEN,
                auth_type="pat",
            )
        return WorkspaceClient(profile=DATABRICKS_PROFILE)
    return WorkspaceClient()


client = artifact_client()


def download_artifact(path: str) -> bytes:
    response = client.files.download(path)
    if response.contents is None:
        raise RuntimeError(f"Artifact is empty: {path}")
    return response.contents.read()


model_bundle = joblib.load(
    io.BytesIO(download_artifact(cfg.SELECTED_MODEL_BUNDLE_PATH))
)
model_metadata = json.loads(
    download_artifact(cfg.SELECTED_MODEL_METADATA_PATH).decode("utf-8")
)

required_bundle_keys = {
    "model",
    "preprocessor",
    "model_name",
    "decision_threshold",
    "risk_thresholds",
    "categorical_features",
    "numerical_features",
    "target",
}
missing_bundle_keys = sorted(required_bundle_keys - set(model_bundle))
if missing_bundle_keys:
    raise ValueError(
        f"Final model bundle is missing keys: {missing_bundle_keys}"
    )

FINAL_MODEL = model_bundle["model"]
PREPROCESSOR = model_bundle["preprocessor"]
MODEL_NAME = str(model_bundle["model_name"])
DECISION_THRESHOLD = float(model_bundle["decision_threshold"])
RISK_THRESHOLDS = dict(model_bundle["risk_thresholds"])
MEDIUM_RISK_THRESHOLD = float(RISK_THRESHOLDS["medium"])
HIGH_RISK_THRESHOLD = float(RISK_THRESHOLDS["high"])
CRITICAL_RISK_THRESHOLD = float(RISK_THRESHOLDS["critical"])
CATEGORICAL_COLUMNS = list(model_bundle["categorical_features"])
NUMERICAL_COLUMNS = list(model_bundle["numerical_features"])
TARGET_COLUMN = str(model_bundle["target"])
MODEL_COLUMNS = CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN

if MODEL_NAME != "XGBoost":
    raise ValueError(f"Expected final XGBoost model; found {MODEL_NAME}.")
if not 0.0 < DECISION_THRESHOLD < 1.0:
    raise ValueError(f"Invalid frozen threshold: {DECISION_THRESHOLD}")
if not (
    0.0 < MEDIUM_RISK_THRESHOLD <= HIGH_RISK_THRESHOLD
    < CRITICAL_RISK_THRESHOLD <= 1.0
):
    raise ValueError(f"Invalid frozen risk thresholds: {RISK_THRESHOLDS}")
if not np.isclose(MEDIUM_RISK_THRESHOLD, DECISION_THRESHOLD):
    raise ValueError(
        "Medium-risk threshold must match the model decision threshold."
    )
if not np.isclose(
    DECISION_THRESHOLD,
    float(model_metadata["decision_threshold"]),
):
    raise ValueError("Bundle and metadata thresholds do not match.")
metadata_risk_thresholds = model_metadata.get("risk_thresholds", {})
for band in ["medium", "high", "critical"]:
    if band not in metadata_risk_thresholds or not np.isclose(
        float(metadata_risk_thresholds[band]),
        float(RISK_THRESHOLDS[band]),
    ):
        raise ValueError(
            f"Bundle and metadata {band}-risk thresholds do not match."
        )
if not spark.catalog.tableExists(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE):
    raise RuntimeError("Run Notebook 09 before Notebook 10.")

shap_importance_pdf = (
    spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
    .orderBy(F.col("MeanAbsSHAP").desc())
    .toPandas()
)
feature_labels = dict(
    zip(
        shap_importance_pdf["FeatureColumn"],
        shap_importance_pdf["Feature"],
    )
)

print(f"Frozen final model: {MODEL_NAME}")
print(f"Configuration: {model_metadata['configuration_id']}")
print(f"Frozen threshold: {DECISION_THRESHOLD:.2f}")
print(
    "Frozen risk bands: "
    f"MEDIUM >= {MEDIUM_RISK_THRESHOLD:.4f}, "
    f"HIGH >= {HIGH_RISK_THRESHOLD:.4f}, "
    f"CRITICAL >= {CRITICAL_RISK_THRESHOLD:.4f}"
)
print(f"Model inputs: {len(MODEL_COLUMNS)}")


Frozen final model: XGBoost
Configuration: baseline-reference
Frozen threshold: 0.20
Frozen risk bands: MEDIUM >= 0.2000, HIGH >= 0.4095, CRITICAL >= 0.4820
Model inputs: 19


## Create the Leakage-Safe Operational Evaluation Sample

The model was trained on January–October. Historical delay rates for every
November–December row are learned only from that earlier period; holdout
outcomes never enter the predictors or prioritization scores. Actual labels are
used only after selection to count captured delays. The same fixed sample limit
and seed used by Notebook 08 keep the evaluation reproducible and manageable.


In [3]:
if not spark.catalog.tableExists(cfg.FEATURES_TABLE):
    raise RuntimeError("Run Notebook 06 before Notebook 10.")

df_features: DataFrame = spark.table(cfg.FEATURES_TABLE)
required_source_columns = (
    set(
        MODEL_COLUMNS
        + list(cfg.MODELING_JOIN_KEY_COLUMNS)
        + [TARGET_COLUMN, DATE_COLUMN]
    )
    - set(cfg.MODEL_HISTORICAL_RATE_COLUMNS)
)
missing_source_columns = sorted(
    required_source_columns - set(df_features.columns)
)
if missing_source_columns:
    raise ValueError(
        f"Feature table is missing columns: {missing_source_columns}"
    )
if not isinstance(df_features.schema[DATE_COLUMN].dataType, T.DateType):
    raise TypeError(f"{DATE_COLUMN} must be a Spark date column.")

df_training_history = df_features.filter(
    F.col(DATE_COLUMN) <= F.lit(cfg.VALIDATION_END_DATE)
)
df_operational = df_features.filter(
    F.col(DATE_COLUMN).between(
        F.lit(cfg.SCORING_START_DATE),
        F.lit(cfg.SCORING_END_DATE),
    )
)

invalid_targets = df_operational.filter(
    F.col(TARGET_COLUMN).isNull()
    | ~F.col(TARGET_COLUMN).isin(0.0, 1.0)
).count()
if invalid_targets:
    raise ValueError(
        f"Operational period contains {invalid_targets:,} invalid targets."
    )

HISTORY_SPECS = [
    ("AIRLINE_HIST_DELAY_RATE", [cfg.AIRLINE_COLUMN]),
    ("ORIGIN_HIST_DELAY_RATE", [cfg.ORIGIN_COLUMN]),
    ("DEST_HIST_DELAY_RATE", [cfg.DESTINATION_COLUMN]),
    (
        "ROUTE_HIST_DELAY_RATE",
        [cfg.ORIGIN_COLUMN, cfg.DESTINATION_COLUMN],
    ),
]
SMOOTHING = float(cfg.HISTORICAL_SMOOTHING_STRENGTH)


def attach_fixed_history(
    training_history: DataFrame,
    rows_to_score: DataFrame,
) -> DataFrame:
    global_rate = float(
        training_history.agg(
            F.avg(TARGET_COLUMN).alias("RATE")
        ).first()["RATE"]
    )
    scored = rows_to_score
    for output_column, group_columns in HISTORY_SPECS:
        history = (
            training_history.groupBy(*group_columns)
            .agg(
                F.count("*").alias("HIST_FLIGHTS"),
                F.sum(TARGET_COLUMN).alias("HIST_DELAYS"),
            )
            .withColumn(
                output_column,
                (
                    F.col("HIST_DELAYS")
                    + SMOOTHING * F.lit(global_rate)
                )
                /
                (F.col("HIST_FLIGHTS") + F.lit(SMOOTHING)),
            )
            .select(*group_columns, output_column)
        )
        scored = scored.join(
            history,
            group_columns,
            "left",
        ).fillna({output_column: global_rate})
    return scored


def uniform_sample(
    frame: DataFrame,
    maximum_rows: int,
    seed: int,
) -> DataFrame:
    row_count = frame.count()
    if row_count <= maximum_rows:
        return frame
    fraction = min(1.0, maximum_rows / row_count * 1.08)
    return frame.sample(False, fraction, seed).limit(maximum_rows)


sampled_operational = uniform_sample(
    df_operational,
    cfg.FINAL_HOLDOUT_MAX_ROWS,
    cfg.RANDOM_SEED + 8001,
)
operational_with_history = attach_fixed_history(
    df_training_history,
    sampled_operational,
)

identifier_columns = list(dict.fromkeys(cfg.MODELING_JOIN_KEY_COLUMNS))
operational_columns = list(
    dict.fromkeys(identifier_columns + MODEL_COLUMNS + [TARGET_COLUMN])
)
operational_pdf = operational_with_history.select(
    *operational_columns
).toPandas()

print(f"Operational rows prepared: {len(operational_pdf):,}")
print(f"Observed delay rate: {operational_pdf[TARGET_COLUMN].mean():.2%}")


Operational rows prepared: 500,000
Observed delay rate: 23.30%


## Score Flights and Build the Operational Queue

The frozen decision threshold converts probabilities into delay alerts. It is
not reused as the High-risk cutoff. Medium begins at the decision threshold,
while High and Critical use the validation P95 and P99 cutoffs frozen by
Notebooks 07 and 08. This keeps classification and dashboard severity separate.


In [4]:
encoded_operational = PREPROCESSOR.transform(
    operational_pdf[MODEL_COLUMNS]
)
delay_probabilities = FINAL_MODEL.predict_proba(
    encoded_operational
)[:, 1]
predicted_delays = (
    delay_probabilities >= DECISION_THRESHOLD
).astype(int)

predictions_pdf = operational_pdf.copy()
predictions_pdf["actual_delay"] = (
    predictions_pdf[TARGET_COLUMN].astype(int)
)
predictions_pdf["predicted_delay"] = predicted_delays
predictions_pdf["delay_probability"] = delay_probabilities
predictions_pdf = predictions_pdf.rename(
    columns={
        cfg.AIRLINE_COLUMN: "airline_code",
        cfg.FLIGHT_NUMBER_COLUMN: "flight_number",
        cfg.ORIGIN_COLUMN: "origin_airport",
        cfg.DESTINATION_COLUMN: "destination_airport",
        cfg.SCHEDULED_DEPARTURE_COLUMN: "scheduled_departure",
        cfg.MONTH_COLUMN: "month_number",
        cfg.TIME_OF_DAY_COLUMN: "departure_window",
        cfg.SEASON_COLUMN: "season",
    }
)


def format_scheduled_time(value: object) -> str:
    if pd.isna(value):
        return ""
    digits = f"{int(value):04d}"
    return f"{digits[:2]}:{digits[2:]}"


predictions_pdf["scheduled_departure_text"] = (
    predictions_pdf["scheduled_departure"].apply(format_scheduled_time)
)
predictions_pdf["flight_label"] = (
    predictions_pdf["airline_code"].astype(str)
    + " "
    + predictions_pdf["flight_number"].astype(int).astype(str)
)
predictions_pdf["shap_main_driver"] = "See Model Insights"

predictions_pdf = add_operational_scores(
    predictions_pdf,
    high_threshold=HIGH_RISK_THRESHOLD,
    critical_threshold=CRITICAL_RISK_THRESHOLD,
    medium_threshold=MEDIUM_RISK_THRESHOLD,
)

print(f"Delay alerts: {predicted_delays.sum():,}")
print(f"Alert rate: {predicted_delays.mean():.2%}")
print(f"High-risk threshold (validation P95): {HIGH_RISK_THRESHOLD:.4f}")
print(
    f"Critical-risk threshold (validation P99): "
    f"{CRITICAL_RISK_THRESHOLD:.4f}"
)
print("Risk-level distribution:")
display(
    predictions_pdf["risk_level"]
    .value_counts(dropna=False)
    .rename_axis("RISK_LEVEL")
    .reset_index(name="FLIGHTS")
)
display(
    predictions_pdf[
        [
            DATE_COLUMN,
            "flight_label",
            "origin_airport",
            "destination_airport",
            "delay_probability",
            "predicted_delay",
            "actual_delay",
            "risk_level",
        ]
    ].head(10)
)


Delay alerts: 238,902
Alert rate: 47.78%
High-risk threshold (validation P95): 0.4095
Critical-risk threshold (validation P99): 0.4820
Risk-level distribution:


,RISK_LEVEL,FLIGHTS
0,LOW,261098
1,MEDIUM,218218
2,HIGH,13889
3,CRITICAL,6795


,FL_DATE,flight_label,origin_airport,destination_airport,delay_probability,predicted_delay,actual_delay,risk_level
0,2025-12-01,AA 1,JFK,LAX,0.135659,0,0,LOW
1,2025-12-01,AA 1002,PVD,ORD,0.172152,0,0,LOW
2,2025-12-01,AA 1003,DEN,MIA,0.253001,1,0,MEDIUM
3,2025-12-01,AA 1006,MIA,LAS,0.359510,1,0,MEDIUM
4,2025-12-01,AA 1008,DFW,ATL,0.303235,1,0,MEDIUM
5,2025-12-01,AA 1014,TUS,DFW,0.203351,1,1,MEDIUM
6,2025-12-01,AA 1015,CHS,DCA,0.161028,0,0,LOW
7,2025-12-01,AA 1019,SJC,DFW,0.102764,0,0,LOW
8,2025-12-01,AA 1020,DTW,MIA,0.120705,0,1,LOW
9,2025-12-01,AA 1023,PHX,IND,0.233807,1,0,MEDIUM


## Add Real Local SHAP Drivers to the Highest-Risk Flights

The dashboard's main driver is calculated from each selected flight's own SHAP
values. It is not inferred by matching flight text to global feature names.
Local SHAP is limited to the highest-risk rows because calculating it for every
operational record is unnecessary and memory intensive.


In [5]:
local_shap_rows = min(
    cfg.PRIORITIZATION_LOCAL_SHAP_ROWS,
    len(predictions_pdf),
)
high_risk_positions = (
    predictions_pdf["delay_probability"]
    .nlargest(local_shap_rows)
    .index
    .to_numpy()
)

explainer = shap.TreeExplainer(FINAL_MODEL)
local_shap_explanation = explainer(
    encoded_operational[high_risk_positions],
    check_additivity=False,
)
encoded_shap_values = np.asarray(local_shap_explanation.values)
if encoded_shap_values.ndim == 3:
    encoded_shap_values = encoded_shap_values[:, :, -1]

encoded_feature_names = [
    str(name) for name in PREPROCESSOR.get_feature_names_out()
]


def original_feature(encoded_name: str) -> str:
    transformer, raw_name = encoded_name.split("__", 1)
    if transformer == "numerical":
        return raw_name
    for column in sorted(CATEGORICAL_COLUMNS, key=len, reverse=True):
        if raw_name == column or raw_name.startswith(f"{column}_"):
            return column
    raise ValueError(
        f"Unable to map encoded feature to an input column: {encoded_name}"
    )


encoded_to_original = [
    original_feature(name) for name in encoded_feature_names
]
aggregated_local_shap = np.zeros(
    (len(high_risk_positions), len(MODEL_COLUMNS)),
    dtype=float,
)
for feature_index, feature_column in enumerate(MODEL_COLUMNS):
    encoded_indexes = [
        index
        for index, original_column in enumerate(encoded_to_original)
        if original_column == feature_column
    ]
    aggregated_local_shap[:, feature_index] = (
        encoded_shap_values[:, encoded_indexes].sum(axis=1)
    )

positive_driver_indexes = np.argmax(aggregated_local_shap, axis=1)
absolute_driver_indexes = np.argmax(
    np.abs(aggregated_local_shap),
    axis=1,
)
has_positive_driver = (
    aggregated_local_shap.max(axis=1) > 0
)
local_driver_indexes = np.where(
    has_positive_driver,
    positive_driver_indexes,
    absolute_driver_indexes,
)
local_driver_labels = [
    feature_labels.get(MODEL_COLUMNS[index], MODEL_COLUMNS[index])
    for index in local_driver_indexes
]
predictions_pdf.loc[
    high_risk_positions,
    "shap_main_driver",
] = local_driver_labels

print(f"Local SHAP drivers calculated for {local_shap_rows:,} flights.")
display(
    predictions_pdf.loc[
        high_risk_positions,
        [
            "flight_label",
            "delay_probability",
            "shap_main_driver",
        ],
    ].head(10)
)


Local SHAP drivers calculated for 20,000 flights.


,flight_label,delay_probability,shap_main_driver
493753,AA 3022,0.797386,Origin airport
283171,YX 5743,0.783011,Month
390567,YX 5743,0.778500,Destination airport
400067,YX 5743,0.774654,Origin airport
283070,YX 4754,0.773792,Month
301243,YX 4754,0.773792,Month
238465,AA 985,0.767096,Origin airport
247579,AA 985,0.767096,Origin airport
492966,AA 1510,0.765445,Origin airport
442798,YX 4803,0.764875,Origin airport


## Compare Prioritization Strategies at Equal Capacity

- **Constrained optimization:** maximizes predicted risk while limiting
  concentration by airline and origin airport.
- **Simple rule:** selects the highest predicted risks without diversification.
- **Random selection:** draws the same number of flights 500 times and reports
  its average and 95% interval.

If constraints prevent the optimizer from filling the requested capacity, both
baselines use the optimizer's actual selected count. This keeps RQ5 fair.


In [6]:
prioritization_pool = (
    predictions_pdf
    .nlargest(
        cfg.PRIORITIZATION_CANDIDATE_POOL_SIZE,
        "delay_probability",
    )
    .copy()
)
print(
    f"Operational candidate queue: "
    f"{len(prioritization_pool):,} highest-risk flights"
)
ranking_tables = []
evaluation_tables = []

for capacity_k in cfg.CAPACITY_K_OPTIONS:
    ranking = build_ranking_table(
        prioritization_pool,
        capacity_k=capacity_k,
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    ranking["capacity_k"] = capacity_k
    ranking_tables.append(ranking)

    evaluation = compare_prioritization_strategies(
        prioritization_pool,
        capacity_k=capacity_k,
        random_seed=cfg.RANDOM_SEED,
        random_repeats=cfg.PRIORITIZATION_RANDOM_REPEATS,
        label_column="actual_delay",
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    evaluation_tables.append(evaluation)

prioritization_results_pdf = pd.concat(
    ranking_tables,
    ignore_index=True,
)
prioritization_evaluation_pdf = pd.concat(
    evaluation_tables,
    ignore_index=True,
)

default_capacity_results = prioritization_evaluation_pdf[
    prioritization_evaluation_pdf["capacity_k"]
    == cfg.DEFAULT_CAPACITY_K
]

display(prioritization_evaluation_pdf)
display(
    prioritization_results_pdf[
        (prioritization_results_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K)
        & prioritization_results_pdf["selected"]
    ].head(25)
)


Operational candidate queue: 10,000 highest-risk flights


,capacity_k,effective_capacity_k,strategy,random_repeats,captured_delays_ci_low,captured_delays_ci_high,random_p_at_least_optimized,population_size,selected_count,total_delayed_flights,captured_delayed_flights,delay_recall,delay_precision,lift_vs_random
0,10,10,Constrained Optimized Selection,500,NaN,NaN,NaN,10000.0,10.0,4528.0,9.000,0.001988,0.900000,1.987633
1,10,10,Top-K Probability Baseline,500,NaN,NaN,NaN,10000.0,10.0,4528.0,7.000,0.001546,0.700000,1.545936
2,10,10,Random Baseline,500,2.0,8.000,0.007984,10000.0,10.0,4528.0,4.546,0.001004,0.454600,1.003975
3,25,25,Constrained Optimized Selection,500,NaN,NaN,NaN,10000.0,25.0,4528.0,16.000,0.003534,0.640000,1.413428
4,25,25,Top-K Probability Baseline,500,NaN,NaN,NaN,10000.0,25.0,4528.0,15.000,0.003313,0.600000,1.325088
5,25,25,Random Baseline,500,7.0,16.000,0.049900,10000.0,25.0,4528.0,11.412,0.002520,0.456480,1.008127
6,50,50,Constrained Optimized Selection,500,NaN,NaN,NaN,10000.0,50.0,4528.0,31.000,0.006846,0.620000,1.369258
7,50,50,Top-K Probability Baseline,500,NaN,NaN,NaN,10000.0,50.0,4528.0,31.000,0.006846,0.620000,1.369258
8,50,50,Random Baseline,500,16.0,30.000,0.017964,10000.0,50.0,4528.0,22.824,0.005041,0.456480,1.008127
9,100,56,Constrained Optimized Selection,500,NaN,NaN,NaN,10000.0,56.0,4528.0,34.000,0.007509,0.607143,1.340863


,FL_DATE,airline_code,flight_number,origin_airport,destination_airport,scheduled_departure,season,departure_window,FLIGHT_DISTANCE_CATEGORY,month_number,DAY_OF_WEEK,DISTANCE,CRS_ELAPSED_TIME,DEP_HOUR,DEP_MINUTE,ARR_HOUR,ARR_MINUTE,IS_WEEKEND,AIRLINE_HIST_DELAY_RATE,ORIGIN_HIST_DELAY_RATE,DEST_HIST_DELAY_RATE,ROUTE_HIST_DELAY_RATE,ARR_DEL15,actual_delay,predicted_delay,delay_probability,scheduled_departure_text,flight_label,shap_main_driver,risk_level,priority_score,recommendation,selected,priority_rank,capacity_k
10000,2025-11-02,AA,3022,LGA,ORD,1855,Fall,Evening,Medium,11,7,733.0,167.0,18,55,20,42,1,0.258603,0.247391,0.242931,0.280353,1,1,1,0.797386,18:55,AA 3022,Origin airport,CRITICAL,80,Immediate Operational Assessment,True,1,25
10001,2025-11-03,YX,5743,LGA,BOS,1630,Fall,Afternoon,Short,11,1,184.0,82.0,16,30,17,52,0,0.203493,0.247391,0.263810,0.242466,0,0,1,0.783011,16:30,YX 5743,Month,CRITICAL,78,Immediate Operational Assessment,True,2,25
10002,2025-11-06,YX,5743,LGA,BOS,1630,Fall,Afternoon,Short,11,4,184.0,82.0,16,30,17,52,0,0.203493,0.247391,0.263810,0.242466,1,1,1,0.778500,16:30,YX 5743,Destination airport,CRITICAL,78,Immediate Operational Assessment,True,3,25
10003,2025-11-13,YX,5743,LGA,BOS,1630,Fall,Afternoon,Short,11,4,184.0,87.0,16,30,17,57,0,0.203493,0.247391,0.263810,0.242466,1,1,1,0.774654,16:30,YX 5743,Origin airport,CRITICAL,77,Immediate Operational Assessment,True,4,25
10007,2025-12-14,AA,985,LGA,ORD,1859,Winter,Evening,Medium,12,7,733.0,171.0,18,59,20,50,1,0.258603,0.247391,0.242931,0.280353,1,1,1,0.767096,18:59,AA 985,Origin airport,CRITICAL,77,Immediate Operational Assessment,True,8,25
10309,2025-11-24,WN,487,AUS,BOS,1420,Fall,Afternoon,Long,11,1,1698.0,235.0,14,20,19,15,0,0.211877,0.221977,0.263810,0.321563,1,1,1,0.659938,14:20,WN 487,Destination airport,CRITICAL,66,Immediate Operational Assessment,True,310,25
10499,2025-11-20,AA,2306,DFW,BOS,1557,Fall,Afternoon,Long,11,4,1562.0,213.0,15,57,20,30,0,0.258603,0.289881,0.263810,0.346580,1,1,1,0.642198,15:57,AA 2306,Destination airport,CRITICAL,64,Immediate Operational Assessment,True,500,25
10513,2025-11-24,AA,379,ORD,BOS,1503,Fall,Afternoon,Medium,11,1,867.0,141.0,15,3,18,24,0,0.258603,0.252181,0.263810,0.297665,1,1,1,0.641330,15:03,AA 379,Destination airport,CRITICAL,64,Immediate Operational Assessment,True,514,25
10601,2025-11-20,YX,5596,SDF,BOS,1615,Fall,Afternoon,Medium,11,4,829.0,151.0,16,15,18,46,0,0.203493,0.189555,0.263810,0.332379,0,0,1,0.634623,16:15,YX 5596,Destination airport,CRITICAL,63,Immediate Operational Assessment,True,602,25
10637,2025-12-28,B6,2624,MCO,DCA,1855,Winter,Evening,Medium,12,7,759.0,136.0,18,55,21,11,1,0.252149,0.244781,0.278333,0.330490,1,1,1,0.632518,18:55,B6 2624,Historical route delay rate,CRITICAL,63,Immediate Operational Assessment,True,638,25


## RQ5 / H5 Result

Full support requires constrained optimization to capture more actual delays
than both repeated random selection and the simple top-risk rule at the same
effective capacity. Beating random but not the simple rule is reported as
partial support rather than being overstated.


In [7]:
rq5_rows = []
for capacity_k, capacity_results in prioritization_evaluation_pdf.groupby(
    "capacity_k"
):
    optimized = capacity_results[
        capacity_results["strategy"]
        == "Constrained Optimized Selection"
    ].iloc[0]
    simple = capacity_results[
        capacity_results["strategy"]
        == "Top-K Probability Baseline"
    ].iloc[0]
    random = capacity_results[
        capacity_results["strategy"] == "Random Baseline"
    ].iloc[0]

    beats_random = (
        optimized["captured_delayed_flights"]
        > random["captured_delayed_flights"]
        and random["random_p_at_least_optimized"] < 0.05
    )
    beats_simple = (
        optimized["captured_delayed_flights"]
        > simple["captured_delayed_flights"]
    )
    if beats_random and beats_simple:
        verdict = "Supported"
    elif beats_random:
        verdict = "Partially supported: beats random, not simple rule"
    else:
        verdict = "Not supported"

    rq5_rows.append(
        {
            "capacity_k": int(capacity_k),
            "effective_capacity_k": int(
                optimized["effective_capacity_k"]
            ),
            "optimized_delays": float(
                optimized["captured_delayed_flights"]
            ),
            "simple_rule_delays": float(
                simple["captured_delayed_flights"]
            ),
            "random_mean_delays": float(
                random["captured_delayed_flights"]
            ),
            "random_95pct_low": float(
                random["captured_delays_ci_low"]
            ),
            "random_95pct_high": float(
                random["captured_delays_ci_high"]
            ),
            "random_p_at_least_optimized": float(
                random["random_p_at_least_optimized"]
            ),
            "beats_random": bool(beats_random),
            "beats_simple_rule": bool(beats_simple),
            "rq5_verdict": verdict,
        }
    )

rq5_summary = pd.DataFrame(rq5_rows)
display(rq5_summary)

default_rq5 = rq5_summary[
    rq5_summary["capacity_k"] == cfg.DEFAULT_CAPACITY_K
].iloc[0]
print(
    f"RQ5 at K={cfg.DEFAULT_CAPACITY_K}: "
    f"{default_rq5['rq5_verdict']}"
)
print(
    "The result describes operational selection performance; "
    "it does not change the predictive model."
)


,capacity_k,effective_capacity_k,optimized_delays,simple_rule_delays,random_mean_delays,random_95pct_low,random_95pct_high,random_p_at_least_optimized,beats_random,beats_simple_rule,rq5_verdict
0,10,10,9.0,7.0,4.546,2.0,8.000,0.007984,True,True,Supported
1,25,25,16.0,15.0,11.412,7.0,16.000,0.049900,True,True,Supported
2,50,50,31.0,31.0,22.824,16.0,30.000,0.017964,True,False,"Partially supported: beats random, not simple rule"
3,100,56,34.0,35.0,24.920,18.0,32.525,0.015968,True,False,"Partially supported: beats random, not simple rule"


RQ5 at K=25: Supported
The result describes operational selection performance; it does not change the predictive model.


## Save Operational and RQ5 Outputs

The three Delta tables below feed Notebook 11, the API, and Streamlit. Existing
tables are replaced so the dashboard cannot mix the old Logistic Regression
pipeline with the final XGBoost pipeline.


In [8]:
def write_pandas_table(
    frame: pd.DataFrame,
    table_name: str,
    delta_path: str,
) -> None:
    clean = frame.copy()
    for column in clean.select_dtypes(include=["object"]).columns:
        clean[column] = clean[column].fillna("").astype(str)
    spark_frame = spark.createDataFrame(clean)
    (
        spark_frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(delta_path)
    )
    (
        spark_frame.writeTo(table_name)
        .using("delta")
        .createOrReplace()
    )


write_pandas_table(
    predictions_pdf,
    cfg.PREDICTIONS_TABLE,
    cfg.PREDICTIONS_DELTA_PATH,
)
write_pandas_table(
    prioritization_results_pdf,
    cfg.PRIORITIZATION_RESULTS_TABLE,
    cfg.PRIORITIZATION_RESULTS_PATH,
)
write_pandas_table(
    prioritization_evaluation_pdf,
    cfg.PRIORITIZATION_EVALUATION_TABLE,
    cfg.PRIORITIZATION_EVALUATION_PATH,
)

print("Notebook 10 completed successfully.")
print(f"Predictions: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization queue: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"RQ5 evaluation: {cfg.PRIORITIZATION_EVALUATION_TABLE}")

gc.collect()


Notebook 10 completed successfully.
Predictions: workspace.default.flight_predictions
Prioritization queue: workspace.default.flight_prioritization_results
RQ5 evaluation: workspace.default.flight_prioritization_evaluation


50